# 1. Setup

3Cv2 two-head BNFR adds blind novel feature recomposition to the four-block two-head 3Cv1. Keep both towers, residual attention, Moving Weighted Sum and the two final primitive classifiers. During development use only training-derived data. Blind recomposition may coincidentally match an official pair without consulting the official list. This run assumes the instructor permits incidental matches.

Inspired by [COMO §3.3](https://ijcai.org/proceedings/2025/0143.pdf). Train on the fused feature using only its source crop and disease labels. There is no Outlier Exposure term, novel joint label or official candidate lookup.

In [ ]:
import gc, json, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_transforms, _make_loader, get_final_test_loaders
from common.evaluate import evaluate, collect_scores, top_k_accuracy, val_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3c_v2_dino_blind_recomposition_only_two_head', 'C'
DEBUG = True
ALLOW_BLIND_RECOMPOSITION = True
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          unfreeze_depth=4, backbone_lr=1e-5,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          dino_revision='main', fusion_attention_heads=4,
          primitive_weight_start=0.75, primitive_weight_end=0.25,
          pseudo_pairs=10, bnfr_weights=[0.25, 0.5])

EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

# 2. Data

Use the shared split, samplers and transforms. For tuning, hold out entire combinations from the training partition. Score their real images from the shared validation partition. Keep both primitives of each pseudo-unseen pair represented in the fit partition. Exclude all original training pairs from synthetic novel targets, including the pseudo-unseen pairs.

In [ ]:
pairs = get_pairs()

def common_loaders():
    cfg.set_seed(cfg.SEED)
    train, val, _, _ = get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
    train.pin_memory = val.pin_memory = False
    return train, val


def pseudo_loaders():
    train, val = common_loaders()
    fit_frame, val_frame = train.dataset.frame, val.dataset.frame
    active = set(fit_frame.pair_row)
    options = sorted(active & set(val_frame.pair_row))
    generator = torch.Generator().manual_seed(cfg.SEED)
    held_out = []
    for index in torch.randperm(len(options), generator=generator).tolist():
        row = options[index]
        remaining = pairs[sorted(active - {row})]
        crop, disease = pairs[row]
        if (remaining[:, 0] == crop).any() and (remaining[:, 1] == disease).any():
            active.remove(row)
            held_out.append(row)
        if len(held_out) == HP['pseudo_pairs']:
            break
    if not held_out:
        raise ValueError('No eligible pseudo-unseen pairs with both primitives in the fit split.')
    keep = sorted(active)
    fit = fit_frame[fit_frame.pair_row.isin(keep)].copy()
    fit['pair_row'] = fit.pair_row.map({row: i for i, row in enumerate(keep)})
    pseudo = val_frame[val_frame.pair_row.isin(held_out)].copy()
    train_transform, eval_transform = get_transforms(HP['norm'])
    fit_loader = _make_loader(fit, train_transform, HP['norm'], HP['batch_size'], True)
    pseudo_loader = _make_loader(pseudo, eval_transform, HP['norm'], HP['batch_size'], False)
    fit_loader.pin_memory = pseudo_loader.pin_memory = False
    return fit_loader, pseudo_loader, pairs[keep], held_out

# 3. Model

The final scorer sums the crop and disease logits from the same fused feature in the shared candidate order. For blind novel examples, the auxiliary heads supervise source crop and disease separately on the fused feature. Real and observed-recomposed examples keep the original 3Cv1 two-head losses.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self, depth):
        self.requires_grad_(False)
        for tower in (self.crop, self.disease):
            tower.blocks[-depth:].requires_grad_(True)
            tower.set_grad_checkpointing(True)

    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))




class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        width, hidden, dim = backbone.width, HP['hidden_dim'], HP['feature_dim']
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)
        self.crop_aux = nn.Linear(dim, cfg.NUM_CROPS)
        self.disease_aux = nn.Linear(dim, cfg.NUM_DISEASES)
        self.attention = nn.MultiheadAttention(dim, HP['fusion_attention_heads'], batch_first=True)
        self.image_fusion = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.GELU())
        self.fused_crop_head = nn.Linear(dim, cfg.NUM_CROPS)
        self.fused_disease_head = nn.Linear(dim, cfg.NUM_DISEASES)

    def features(self, images):
        crop, disease = self.backbone(images)
        return self.crop_projection(crop), self.disease_projection(disease)

    def fuse(self, crop, disease):
        combined = crop + disease
        tokens = torch.stack((crop + combined, disease + combined), dim=1)
        attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
        residuals = torch.stack((disease, crop), dim=1)
        return self.image_fusion((attended + residuals).flatten(1))

    def score(self, crop, disease, candidate_pairs):
        image = self.fuse(crop, disease)
        crop_ids, disease_ids = candidate_pairs.unbind(1)
        return self.fused_crop_head(image)[:, crop_ids] + self.fused_disease_head(image)[:, disease_ids]


def sample_observed_pairs(crops, diseases, pair_rows, generator):
    """Sample at most one batch's worth of admissible cross-image combinations."""
    rows = pair_rows[crops[:, None], diseases[None, :]]
    eligible = (rows >= 0) & ~torch.eye(len(crops), dtype=torch.bool, device=rows.device)
    crop_sources, disease_sources = eligible.nonzero(as_tuple=True)
    order = torch.randperm(len(crop_sources), generator=generator)[:len(crops)].to(rows.device)
    crop_sources, disease_sources = crop_sources[order], disease_sources[order]
    return crop_sources, disease_sources, rows[crop_sources, disease_sources]


def training_loss(model, images, labels, candidates, pair_rows, generator, primitive_weight,
                  settings, observed, novel_generator):
    crop, disease = model.features(images)
    crops, diseases = candidates[labels].unbind(1)
    smoothing = HP['label_smoothing']
    primitive = (F.cross_entropy(model.crop_aux(crop), crops, label_smoothing=smoothing)
                 + F.cross_entropy(model.disease_aux(disease), diseases, label_smoothing=smoothing)) / 2
    joint = F.cross_entropy(model.score(crop, disease, candidates), labels, label_smoothing=smoothing)
    src_crop, src_disease, targets = sample_observed_pairs(crops, diseases, pair_rows, generator)
    if targets.numel():
        synthetic = model.score(crop[src_crop], disease[src_disease], candidates)
        joint = (joint + F.cross_entropy(synthetic, targets, label_smoothing=smoothing)) / 2
    loss = primitive_weight * primitive + (1 - primitive_weight) * joint
    novel_count = 0
    if settings['bnfr']:
        extra, novel_count = novel_primitive_loss(model, crop, disease, crops, diseases,
                                                 observed, novel_generator, settings)
        loss = loss + extra
    return loss, novel_count


def make_model(depth, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(depth)
    cfg.set_seed(cfg.SEED)
    model = Model(backbone).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, dict(dino=backbone.metadata)



def sample_novel(crops, diseases, observed, generator):
    eligible = ~observed[crops[:, None], diseases[None, :]]
    eligible.fill_diagonal_(False)
    left, right = eligible.nonzero(as_tuple=True)
    sample = torch.randperm(len(left), generator=generator)[:len(crops)].to(left.device)
    return left[sample], right[sample]


def novel_primitive_loss(model, crop, disease, crops, diseases, observed, generator, settings):
    left, right = sample_novel(crops, diseases, observed, generator)
    if not left.numel():
        return crop.sum() * 0, 0
    fused = model.fuse(crop[left], disease[right])
    primitive = (F.cross_entropy(model.crop_aux(fused), crops[left])
                 + F.cross_entropy(model.disease_aux(fused), diseases[right])) / 2

    return settings['bnfr'] * primitive, left.numel()


def make_predict(model, candidate_pairs):
    candidates = candidate_pairs.to(cfg.DEVICE)

    @torch.no_grad()
    def predict(images):
        model.eval()
        return model.score(*model.features(images.to(cfg.DEVICE)), candidates)
    return predict

# 4. Training

Tune only the BNFR weight on training-derived pseudo-unseen validation. Refit from the same pretrained revisions on the full shared training split for 20 epochs. As in the two-head 3Cv1 control, select the final checkpoint by shared validation Top-1, keeping the earlier epoch on ties.

In [ ]:
def fit_model(depth, train_loader, candidates, settings, revisions, val_loader=None):
    if settings['bnfr'] and not ALLOW_BLIND_RECOMPOSITION:
        raise ValueError('Blind novel recomposition is disabled by the protocol gate.')
    cfg.set_seed(cfg.SEED)
    model, metadata = make_model(depth, revisions)
    candidates = candidates.to(cfg.DEVICE)
    pair_rows = torch.full((cfg.NUM_CROPS, cfg.NUM_DISEASES), -1, device=cfg.DEVICE)
    pair_rows[candidates[:, 0], candidates[:, 1]] = torch.arange(len(candidates), device=cfg.DEVICE)
    observed = torch.zeros_like(pair_rows, dtype=torch.bool)
    observed[pairs[:, 0], pairs[:, 1]] = True
    generator = torch.Generator().manual_seed(cfg.SEED)
    novel_generator = torch.Generator().manual_seed(cfg.SEED)
    groups = {'backbone': [], 'crop': [], 'disease': []}
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            key = 'backbone' if name.startswith('backbone.') else 'disease' if 'disease' in name else 'crop'
            groups[key].append(parameter)
    optimizer = torch.optim.AdamW(
        [dict(params=groups[key], lr=HP[key + '_lr']) for key in groups],
        weight_decay=HP['weight_decay'])
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best = None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        model.backbone.eval()
        for tower in (model.backbone.crop, model.backbone.disease):
            tower.blocks[-depth:].train()
        progress = (epoch - 1) / (EPOCHS - 1) if EPOCHS > 1 else 0.5
        primitive_weight = HP['primitive_weight_start'] + progress * (
            HP['primitive_weight_end'] - HP['primitive_weight_start'])
        total_loss = synthetic_count = samples = 0
        for images, _, _, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            value, synthetic = training_loss(model, images.to(cfg.DEVICE), labels.to(cfg.DEVICE),
                                             candidates, pair_rows, generator, primitive_weight,
                                             settings, observed, novel_generator)
            if not torch.isfinite(value):
                raise RuntimeError('Nonfinite training loss.')
            value.backward()
            optimizer.step()
            samples += len(labels)
            total_loss += value.item() * len(labels)
            synthetic_count += synthetic
        schedule.step()
        print(f"epoch {epoch}: loss={total_loss / samples:.4f}, novel examples={synthetic_count}")
        if val_loader is not None:
            accuracy = val_accuracy(make_predict(model, pairs), val_loader)
            print(f"validation Top-1: {accuracy:.2f}%")
            if best is None or accuracy > best['val_top1']:
                best = dict(epoch=epoch, val_top1=accuracy,
                            state={key: value.detach().to('cpu', copy=True)
                                   for key, value in model.state_dict().items()})
    return model, metadata, best



def train_bnfr(depth):
    chosen = revisions = None
    for weight in HP['bnfr_weights']:
        fit, pseudo, fit_pairs, held_out = pseudo_loaders()
        settings = dict(bnfr=weight)
        model, revisions, _ = fit_model(depth, fit, fit_pairs, settings, revisions)
        scores, labels = collect_scores(make_predict(model, pairs), pseudo)
        trial = dict(settings, pseudo_top1=top_k_accuracy(scores, labels, 1))
        print(trial)
        if chosen is None or trial['pseudo_top1'] > chosen['pseudo_top1']:
            chosen = trial
        del model
    train, val = common_loaders()
    model, revisions, best = fit_model(depth, train, pairs, chosen, revisions, val)
    identity = dict(hp=HP, debug=DEBUG, model_name=MODEL_NAME)
    selection = dict(identity=identity, settings=chosen, pseudo_pair_rows=held_out,
                     best_epoch=best["epoch"], best_val=best["val_top1"],
                     train_images=train.dataset.frame.image.tolist(),
                     validation_images=val.dataset.frame.image.tolist())
    torch.save(dict(selection=selection, pairs=pairs, metadata=revisions,
                    state_dict=best["state"]), OUT / f'{MODEL_NAME}.pt')
    (OUT / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf8')
    print('Selected:', chosen, 'epoch:', best['epoch'], 'validation Top-1:', best['val_top1'])


if ALLOW_BLIND_RECOMPOSITION:
    OUT.mkdir(parents=True, exist_ok=False)
    set_reentrant_ckpt(False)
    train_bnfr(HP['unfreeze_depth'])
else:
    print('Novel recomposition is disabled pending protocol clearance.')

# 5. Evaluation

The shared evaluator ranks all 333 candidates from the unchanged joint scorer. Enable final evaluation only after refitting is complete and all choices are frozen.

In [ ]:
RUN_FINAL_EVALUATION = False
report = None
if RUN_FINAL_EVALUATION:
    if DEBUG or not ALLOW_BLIND_RECOMPOSITION:
        raise ValueError('Official evaluation requires an enabled full non-DEBUG run.')
    selection = json.loads((OUT / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(OUT / f'{MODEL_NAME}.pt', map_location='cpu', weights_only=True)
    expected = dict(hp=HP, debug=False, model_name=MODEL_NAME)
    if (selection['identity'] != expected or selection != checkpoint['selection']
            or not torch.equal(checkpoint['pairs'], pairs)):
        raise ValueError('Checkpoint does not match the frozen selection.')
    train, val = common_loaders()
    if (selection['train_images'] != train.dataset.frame.image.tolist()
            or selection['validation_images'] != val.dataset.frame.image.tolist()):
        raise ValueError('Shared training/validation membership changed.')
    model, metadata = make_model(HP['unfreeze_depth'], checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    for tower in (model.backbone.crop, model.backbone.disease):
        tower.set_grad_checkpointing(False)
    final_pairs = get_pairs(final_evaluation=True)
    seen, unseen = get_final_test_loaders(HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    predict = make_predict(model, final_pairs)
    RESULTS = cfg.RESULTS_DIR / OUT.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        report = evaluate(predict, seen, unseen, MODEL_NAME, dict(HP, selection=selection), robust=True)
    finally:
        cfg.RESULTS_DIR = previous_results

# 6. Save

Keep the checkpoint, tuning manifest and shared evaluation metrics and confusion files with the original control's results.

In [ ]:
if report is not None:
    fields = ['seen_top1', 'seen_top5', 'unseen_top1', 'unseen_top5', 'harmonic_mean']
    display(pd.DataFrame([{key: report[key] for key in fields}]))
    print('Results:', RESULTS)
if (OUT / 'selection.json').exists():
    print('Checkpoint and selection:', OUT)